# Delta hedging on historical NVDA closes

Signed hedging error is terminal financed P&L per 100 option units. Absolute error measures distance from zero. MAE is its average and RMSE weights large errors more heavily. P05 is the fifth percentile of signed P&L.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'modules' / 'engine.py').is_file()), None)
if root is None:
    raise RuntimeError('Start Jupyter from the VolatilityLab repository.')
sys.path.insert(0, str(root))
from modules.engine import historical_nvda_prices, historical_episode, simulate_path, hedge

def summarize(results):
    return (results.groupby(['Fee case', 'Strategy'], sort=False)['Signed error ($)']
            .agg(Episodes='size', Mean='mean', MAE=lambda x: x.abs().mean(),
                 RMSE=lambda x: np.sqrt(x.pow(2).mean()),
                 P05=lambda x: x.quantile(.05)).round(2))

def review(results, baseline):
    display(summarize(results))
    net = results.loc[results['Fee case'].eq('Net')].copy()
    net['Absolute error ($)'] = net['Signed error ($)'].abs()
    strategies = net['Strategy'].drop_duplicates().tolist()
    terminal = px.box(
        net, x='Strategy', y='Signed error ($)', color='Strategy',
        category_orders={'Strategy': strategies}, points='outliers',
        title='After-fee terminal P&L by strategy',
    )
    terminal.update_xaxes(
        type='category', tickmode='array', tickvals=strategies,
        ticktext=strategies, tickangle=0, automargin=True, showticklabels=True,
    )
    terminal.update_layout(boxmode='overlay', legend_title_text='Strategy')
    terminal.add_hline(y=0).show()
    px.ecdf(net, x='Absolute error ($)', color='Strategy',
            title='Fraction of episodes within each dollar error threshold').show()

    paired = net.pivot(index='Path', columns='Strategy', values='Absolute error ($)')
    gains = paired.drop(columns=baseline).rsub(paired[baseline], axis=0)
    gain_long = gains.melt(var_name='Strategy', value_name='Improvement ($)')
    px.box(gain_long, x='Strategy', y='Improvement ($)',
           title=f'Paired absolute-error improvement versus {baseline}').update_xaxes(type='category').add_hline(y=0).show()

    metrics = summarize(results).reset_index()
    px.bar(metrics, x='Strategy', y='MAE', color='Fee case', barmode='group',
           text_auto='.2f', hover_data=['RMSE', 'Mean', 'P05', 'Episodes'],
           labels={'MAE': 'Mean absolute hedging error ($)'},
           title='Main result · mean absolute error before and after fees').update_xaxes(
               type='category', tickmode='array', tickvals=metrics['Strategy'].unique(),
               ticktext=metrics['Strategy'].unique(), automargin=True).show()
    print(f'Mean paired improvement versus {baseline}, after fees ($):')
    display(gains.mean().round(2))


## Load and explore NVDA

The default sample ends at the latest completed close available from Yahoo Finance. Set an explicit cutoff to rerun the same evaluation period. The charts show the training history and how volatility varies through time.

In [ ]:
CUTOFF = None  # Optional YYYY-MM-DD date
prices, fetched_at = historical_nvda_prices()
if CUTOFF is not None:
    prices = prices.loc[:CUTOFF]
returns = np.log(prices).diff().dropna()
px.line(prices, labels={'value': 'NVDA close ($)'}, title='NVDA closing prices').show()
px.histogram(x=returns*100, nbins=60, labels={'x': 'Daily log return (%)'},
             title='Distribution of NVDA daily returns').show()
px.line((returns.pow(2).rolling(20).mean()*252).pow(.5)*100,
        labels={'value': 'Annualized volatility (%)'},
        title='Trailing 20-session realized volatility').show()


## Walk forward through five-session episodes

The default evaluates 24 non-overlapping return windows. Each strategy re-estimates volatility for the remaining life using closes through the current decision date. All strategies use the same observed stock moves, hypothetical at-the-money call, and opening premium based on trailing historical volatility. Option fills and premiums are modeled. The volatility oracle uses the annualized RMS of the future returns remaining in each episode (floored at 0.01% for flat paths). It uses hindsight, shares the same opening premium and fees, and follows the same delta rule.

In [ ]:
WINDOWS = 24
COST = .001
ends = list(range(len(prices), 319, -5))[:WINDOWS]
if not ends:
    raise ValueError('At least 320 completed closes are required.')

rows = []
forecasts = []
for end in reversed(ends):
    episode = historical_episode(prices.iloc[:end], cost=COST, include_oracle=True)
    forecasts.append(episode['forecasts'].iloc[[0]].assign(
        Realized=np.sqrt(np.mean(np.diff(np.log(episode['path']))**2)*252)*100
    ))
    for row in episode['summary'].to_dict('records'):
        for case, column in [('Net', 'Terminal P&L ($)'), ('Gross', 'P&L before costs ($)')]:
            rows.append({'Path': episode['path'].index[-1], 'Strategy': row['Strategy'],
                         'Fee case': case, 'Signed error ($)': row[column]})
historical = pd.DataFrame(rows)
initial = pd.concat(forecasts, ignore_index=True).set_index('Decision date')
px.line(initial[['Historical', 'GARCH', 'XGBoost', 'Realized']], markers=True,
        labels={'value': 'Annualized volatility (%)', 'variable': 'Series'},
        title='Opening forecasts versus subsequent five-session realized volatility').show()


## Compare hedging errors

Positive paired improvement means smaller absolute error than the historical-volatility hedge on the same NVDA window. The final chart shows whether average error improves after trading fees.

In [ ]:
review(historical, 'Historical')

This sample covers one stock over a limited period. Compare the paired results across additional cutoffs, market regimes, and fee levels before drawing a broader conclusion.